# SimpleLLM V0.15 — GPT-style Decoder-Only Transformer

**Изменения относительно V0.14:**
- **N-gram blocking** в генерации: запрет повторения триграмм → больше никаких циклов
- **Gradient clipping** (clipnorm=1.0): стабилизирует обучение глубоких моделей
- **LR Warmup** (1000 шагов): плавное начало обучения, затем cosine decay
- **+6 датасетов** (Джейн Эйр, Одиссея, Граф Монте-Кристо, Гулливер, Робинзон Крузо, 1984-эпоха)
- **Улучшенная генерация**: top-p (nucleus) sampling + жёстче repetition penalty
- Все остальные улучшения V0.14 сохранены (BPE, Pre-Norm, GELU, Weight Tying)

In [ ]:
# ========================== ЗАВИСИМОСТИ ==========================
# sentencepiece — BPE токенизатор (тот же алгоритм что в GPT-2, LLaMA, Mistral)
!pip install -q sentencepiece

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
import sentencepiece as spm
import os, urllib.request, tempfile

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ========================== ГИПЕРПАРАМЕТРЫ ==========================
# Оптимизированы под ~160K строк из 26 датасетов (9 жанров).
# Стратегия: тонкая но глубокая модель (6 блоков × 256 dim) вместо
# широкой и мелкой (4 блока × 512 dim). Лучше обобщает при таком объёме данных.

MAX_VOCAB = 8000         # BPE гарантирует точный размер (в отличие от WordPiece)
CONTEXT_WIN = 128        # Увеличено 50→128: модель видит больше контекста → связнее текст
EMBED_DIM = 256          # Уменьшено 512→256: меньше параметров → меньше переобучение
HEADS = 8                # 256 / 8 = 32 на голову (достаточно для нашего масштаба)
FEED_FORWARD = 1024      # 4 × EMBED_DIM (стандартное соотношение)
TRANSFORMER_BLOCKS = 6   # Увеличено 4→6: глубже модель → лучше выучивает паттерны
DROPOUT_RATE = 0.15      # Увеличено 0.1→0.15: сильнее регуляризация

BATCH_SIZE = 64          # Tesla P100 16GB: можно до 128 при EMBED=256
EPOCHS = 30              # Достаточно с EarlyStopping (patience=5)
LEARNING_RATE = 5e-4     # Снижено 1e-3→5e-4: стабильнее обучение Pre-Norm
MIN_LR = 1e-5            # Минимальный LR (конец cosine decay)
VALIDATION_SPLIT = 0.1   # 10% данных на валидацию

MAX_TRAIN_LINES = 300000 # Используем ВСЕ доступные строки
MIN_LINE_LENGTH = 15     # Фильтрация мусора

# Оценка параметров модели:
# Embedding: 8000 × 256 ≈ 2M
# 6 блоков × (Attn + FFN): 6 × (256² × 4 + 256 × 1024 × 2) ≈ 4.7M
# Итого: ~6.7M параметров (было ~12M в V0.13 → x1.8 меньше)
print(f"Ожидаемые параметры: ~{(MAX_VOCAB * EMBED_DIM + TRANSFORMER_BLOCKS * (4 * EMBED_DIM**2 + 2 * EMBED_DIM * FEED_FORWARD)) / 1e6:.1f}M")

In [ ]:
# ========================== ЗАГРУЗКА ДАННЫХ (32 датасета, 10 жанров) ==========================
import re

DATASETS = {
    # =================== ДРАМАТУРГИЯ / ПОЭЗИЯ ===================
    'shakespeare': {
        'url': 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
        'filename': 'shakespeare.txt'
    },
    'tiny_shakespeare': {
        'url': 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'filename': 'tiny_shakespeare.txt'
    },

    # =================== РЕЛИГИЯ / ФИЛОСОФИЯ ===================
    'bible_kjv': {
        'url': 'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'filename': 'bible_kjv.txt'
    },
    'plato_republic': {
        'url': 'https://www.gutenberg.org/cache/epub/1497/pg1497.txt',
        'filename': 'plato_republic.txt'
    },

    # =================== НАУКА ===================
    'darwin_origin': {
        'url': 'https://www.gutenberg.org/cache/epub/1228/pg1228.txt',
        'filename': 'darwin_origin.txt'
    },
    'einstein_relativity': {
        'url': 'https://www.gutenberg.org/cache/epub/5001/pg5001.txt',
        'filename': 'einstein_relativity.txt'
    },

    # =================== ГОТИКА / ХОРРОР ===================
    'poe': {
        'url': 'https://www.gutenberg.org/cache/epub/2147/pg2147.txt',
        'filename': 'poe.txt'
    },
    'frankenstein': {
        'url': 'https://www.gutenberg.org/cache/epub/84/pg84.txt',
        'filename': 'frankenstein.txt'
    },
    'dracula': {
        'url': 'https://www.gutenberg.org/cache/epub/345/pg345.txt',
        'filename': 'dracula.txt'
    },

    # =================== ДЕТЕКТИВЫ ===================
    'sherlock': {
        'url': 'https://www.gutenberg.org/cache/epub/1661/pg1661.txt',
        'filename': 'sherlock.txt'
    },

    # =================== ПРИКЛЮЧЕНИЯ / ФАНТАСТИКА ===================
    'alice': {
        'url': 'https://www.gutenberg.org/cache/epub/11/pg11.txt',
        'filename': 'alice.txt'
    },
    'around_the_world': {
        'url': 'https://www.gutenberg.org/cache/epub/103/pg103.txt',
        'filename': 'around_the_world.txt'
    },
    'war_of_worlds': {
        'url': 'https://www.gutenberg.org/cache/epub/36/pg36.txt',
        'filename': 'war_of_worlds.txt'
    },
    'time_machine': {
        'url': 'https://www.gutenberg.org/cache/epub/35/pg35.txt',
        'filename': 'time_machine.txt'
    },
    'treasure_island': {
        'url': 'https://www.gutenberg.org/cache/epub/120/pg120.txt',
        'filename': 'treasure_island.txt'
    },
    'twenty_thousand_leagues': {
        'url': 'https://www.gutenberg.org/cache/epub/164/pg164.txt',
        'filename': 'twenty_thousand_leagues.txt'
    },
    'gulliver': {
        'url': 'https://www.gutenberg.org/cache/epub/829/pg829.txt',
        'filename': 'gulliver.txt'
    },
    'robinson_crusoe': {
        'url': 'https://www.gutenberg.org/cache/epub/521/pg521.txt',
        'filename': 'robinson_crusoe.txt'
    },

    # =================== РОМАНЫ / ПРОЗА ===================
    'pride_prejudice': {
        'url': 'https://www.gutenberg.org/cache/epub/1342/pg1342.txt',
        'filename': 'pride_prejudice.txt'
    },
    'moby_dick': {
        'url': 'https://www.gutenberg.org/cache/epub/2701/pg2701.txt',
        'filename': 'moby_dick.txt'
    },
    'tom_sawyer': {
        'url': 'https://www.gutenberg.org/cache/epub/74/pg74.txt',
        'filename': 'tom_sawyer.txt'
    },
    'huck_finn': {
        'url': 'https://www.gutenberg.org/cache/epub/76/pg76.txt',
        'filename': 'huck_finn.txt'
    },
    'great_expectations': {
        'url': 'https://www.gutenberg.org/cache/epub/1400/pg1400.txt',
        'filename': 'great_expectations.txt'
    },
    'tale_two_cities': {
        'url': 'https://www.gutenberg.org/cache/epub/98/pg98.txt',
        'filename': 'tale_two_cities.txt'
    },
    'dorian_gray': {
        'url': 'https://www.gutenberg.org/cache/epub/174/pg174.txt',
        'filename': 'dorian_gray.txt'
    },
    'heart_of_darkness': {
        'url': 'https://www.gutenberg.org/cache/epub/219/pg219.txt',
        'filename': 'heart_of_darkness.txt'
    },
    'jane_eyre': {
        'url': 'https://www.gutenberg.org/cache/epub/1260/pg1260.txt',
        'filename': 'jane_eyre.txt'
    },
    'monte_cristo': {
        'url': 'https://www.gutenberg.org/cache/epub/1184/pg1184.txt',
        'filename': 'monte_cristo.txt'
    },

    # =================== ПОЛИТИКА / ИСТОРИЯ ===================
    'the_prince': {
        'url': 'https://www.gutenberg.org/cache/epub/1232/pg1232.txt',
        'filename': 'the_prince.txt'
    },
    'art_of_war': {
        'url': 'https://www.gutenberg.org/cache/epub/132/pg132.txt',
        'filename': 'art_of_war.txt'
    },

    # =================== УТОПИЯ / АНТИУТОПИЯ ===================
    'utopia': {
        'url': 'https://www.gutenberg.org/cache/epub/2130/pg2130.txt',
        'filename': 'utopia.txt'
    },

    # =================== ЭПОС / МИФОЛОГИЯ ===================
    'odyssey': {
        'url': 'https://www.gutenberg.org/cache/epub/1727/pg1727.txt',
        'filename': 'odyssey.txt'
    },
    'iliad': {
        'url': 'https://www.gutenberg.org/cache/epub/6130/pg6130.txt',
        'filename': 'iliad.txt'
    },
}

def clean_line(line):
    """Очищает строку от мусорного Unicode, оставляя только ASCII + базовую пунктуацию."""
    line = line.strip()
    # Заменяем фигурные кавычки, em-dash, en-dash и прочий Unicode на ASCII
    line = line.replace('\u2018', "'").replace('\u2019', "'")  # ' '
    line = line.replace('\u201c', '"').replace('\u201d', '"')  # " "
    line = line.replace('\u2014', '--').replace('\u2013', '-') # — –
    line = line.replace('\u2026', '...')                        # …
    # Убираем все оставшиеся не-ASCII символы
    line = re.sub(r'[^\x20-\x7E]', '', line)
    # Убираем множественные пробелы
    line = re.sub(r'\s+', ' ', line).strip()
    return line

# Директория для кэша
cache_dir = os.path.join(os.path.expanduser('~'), '.keras', 'datasets', 'llm_corpus')
os.makedirs(cache_dir, exist_ok=True)

all_lines = []
dataset_stats = {}

for name, info in DATASETS.items():
    filepath = os.path.join(cache_dir, info['filename'])
    try:
        if not os.path.exists(filepath):
            print(f"  Скачиваю {name}...")
            urllib.request.urlretrieve(info['url'], filepath)

        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.read().split('\n')

        # Фильтруем + чистим каждую строку
        filtered = []
        for line in lines:
            cleaned = clean_line(line)
            if (len(cleaned) > MIN_LINE_LENGTH
                and not cleaned.startswith('***')
                and 'gutenberg' not in cleaned.lower()
                and 'project gutenberg' not in cleaned.lower()):
                filtered.append(cleaned)

        all_lines.extend(filtered)
        dataset_stats[name] = len(filtered)
        print(f"  ✓ {name}: {len(filtered):,} строк")

    except Exception as e:
        print(f"  ✗ {name}: не удалось загрузить — {e}")

# Перемешиваем
np.random.shuffle(all_lines)

# Добавляем end-of-sequence токен
train_text = [line + ' endseq' for line in all_lines[:MAX_TRAIN_LINES]]

print(f"\n{'='*60}")
print(f"  Всего строк в корпусе:    {len(all_lines):,}")
print(f"  Строк для обучения:       {len(train_text):,}")
print(f"  Датасетов загружено:      {len(dataset_stats)}/{len(DATASETS)}")
print(f"{'='*60}")
print(f"  Пример: {train_text[0][:100]}...")

In [ ]:

# ========================== ТОКЕНИЗАЦИЯ (SentencePiece BPE) ==========================
# ИСПРАВЛЕНО: устраняем зависание spm.SentencePieceTrainer.train()
# Причина зависания: строки-параграфы из Gutenberg могут быть >1024 символов,
# SentencePiece "застревает" в фазе merge при обработке очень длинных строк.
# Решение: жёстко обрезаем каждую строку до 512 символов ДО сохранения корпуса.

import time

# ---- Шаг 1: Перезаписываем корпус с обрезкой строк до 512 символов ----
MAX_LINE_CHARS = 512  # Гарантированно меньше max_sentence_length=1024

corpus_path = os.path.join(cache_dir, 'corpus_for_spm.txt')
lines_written = 0
with open(corpus_path, 'w', encoding='utf-8') as f:
    for line in train_text:
        # Обрезаем до MAX_LINE_CHARS — исключаем длинные параграфы без переносов
        truncated = line[:MAX_LINE_CHARS]
        f.write(truncated + '\n')
        lines_written += 1

print(f"Корпус сохранён: {corpus_path}")
print(f"Строк записано:  {lines_written:,} (обрезаны до {MAX_LINE_CHARS} символов)")
print(f"Размер файла:    {os.path.getsize(corpus_path) / 1024 / 1024:.1f} MB")

# ---- Шаг 2: Удаляем старую модель чтобы не использовать кэш ----
spm_model_prefix = os.path.join(cache_dir, 'spm_bpe_v015')
for ext in ['.model', '.vocab']:
    path = spm_model_prefix + ext
    if os.path.exists(path):
        os.remove(path)
        print(f"  Удалён старый файл: {path}")

# ---- Шаг 3: Обучаем BPE токенизатор ----
print(f"\nОбучаем BPE токенизатор (vocab_size={MAX_VOCAB})...")
print(f"input_sentence_size=50000 (выборка из корпуса, не весь файл — быстрее)")
print(f"byte_fallback=True (никаких [UNK] — редкие байты кодируются напрямую)")

EXPECTED_MAX_SECONDS = 120  # Если дольше — что-то не так
start_time = time.time()

spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix=spm_model_prefix,
    vocab_size=MAX_VOCAB,
    model_type='bpe',

    # КЛЮЧЕВЫЕ ИСПРАВЛЕНИЯ:
    input_sentence_size=50000,     # Снижено 100K→50K: меньше строк = быстрее сходимость BPE
    shuffle_input_sentence=True,   # Случайная выборка из корпуса (не первые N строк)
    character_coverage=0.9995,     # Покрытие символов (ASCII корпус → достаточно)
    byte_fallback=True,            # Редкие символы кодируются байтами → 0 [UNK] токенов
    max_sentence_length=1024,      # Строки уже обрезаны до 512 — с запасом

    # Зарезервированные токены (порядок важен!)
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece='[PAD]',
    unk_piece='[UNK]',
    bos_piece='[START]',
    eos_piece='[END]',

    # Пользовательские символы
    user_defined_symbols=['endseq'],

    # Производительность
    num_threads=os.cpu_count(),
    train_extremely_large_corpus=False,  # Наш корпус не "extremely large" (< 1M строк)
)

elapsed = time.time() - start_time
print(f"\nГотово за {elapsed:.1f} секунд!")

# ---- Предупреждение если обучение заняло слишком долго ----
if elapsed > EXPECTED_MAX_SECONDS:
    print(f"⚠ ВНИМАНИЕ: обучение заняло >{EXPECTED_MAX_SECONDS}с — возможна проблема с корпусом.")
    print(f"  Проверьте: нет ли строк >512 символов или нестандартных символов.")
else:
    print(f"✓ Время обучения в норме (< {EXPECTED_MAX_SECONDS}с)")

# ---- Шаг 4: Загружаем обученную модель ----
sp = spm.SentencePieceProcessor()
sp.load(spm_model_prefix + '.model')

actual_vocab_size = sp.get_piece_size()
print(f"\nРазмер словаря: {actual_vocab_size:,} (запрошено: {MAX_VOCAB:,})")

# ---- Шаг 5: Проверка токенизации ----
test_words = ['Alice', 'electromagnetic', 'Shakespeare', 'natural selection', 'I am the king']
print(f"\nПроверка токенизации:")
for word in test_words:
    tokens = sp.encode(word, out_type=str)
    ids    = sp.encode(word, out_type=int)
    print(f"  '{word}' → {tokens} {ids}")

# Проверка отсутствия [UNK] (byte_fallback должен устранить их полностью)
unk_count = sum(1 for line in train_text[:1000]
                for tid in sp.encode(line, out_type=int)
                if tid == sp.unk_id())
print(f"\n[UNK] токенов в первых 1000 строках: {unk_count} (должно быть 0 при byte_fallback=True)")


In [ ]:

# ========================== SPTokenizer (обёртка для tf.data) ==========================
# SentencePiece возвращает списки Python переменной длины.
# Нам нужны numpy-массивы ФИКСИРОВАННОЙ длины для батчирования в TensorFlow.
# encode_batch работает чанками по 10000 строк — показывает прогресс и не "зависает".

class SPTokenizer:
    """Обёртка над SentencePiece для совместимости с tf.data пайплайном.
    Обеспечивает паддинг до фиксированной длины и батч-обработку."""

    def __init__(self, sp_model, seq_length):
        self.sp = sp_model
        self.seq_length = seq_length
        self.pad_id = sp_model.pad_id()  # 0 → [PAD]

    def encode(self, text):
        """Кодирует одну строку → список int длины seq_length с паддингом справа.

        Args:
            text: строка или bytes (tf.data может передавать bytes)
        Returns:
            list[int] длины self.seq_length (усечение + паддинг нулями)
        """
        # tf.data внутри tf.py_function передаёт bytes — декодируем
        if isinstance(text, (bytes, np.bytes_)):
            text = text.decode('utf-8')
        elif not isinstance(text, str):
            text = str(text)

        ids = self.sp.encode(text, out_type=int)

        # Усекаем если строка длиннее окна
        ids = ids[:self.seq_length]

        # Паддим нулями справа до фиксированной длины
        ids = ids + [self.pad_id] * (self.seq_length - len(ids))

        return ids  # Всегда ровно self.seq_length элементов

    def encode_batch(self, texts):
        """Кодирует список строк → numpy array формы (N, seq_length).

        Обработка идёт ЧАНКАМИ по 10000 строк с выводом прогресса.
        Это исключает "зависание" при большом корпусе (~160K строк).

        Args:
            texts: list[str] | list[bytes] | tf.Tensor
        Returns:
            np.ndarray dtype=int32, shape=(len(texts), seq_length)
        """
        # Конвертируем tf.Tensor в Python-список если нужно
        if hasattr(texts, 'numpy'):
            texts = [
                t.decode('utf-8') if isinstance(t, bytes) else str(t)
                for t in texts.numpy()
            ]
        # Конвертируем bytes → str если пришёл список байтов
        elif len(texts) > 0 and isinstance(texts[0], (bytes, np.bytes_)):
            texts = [t.decode('utf-8') for t in texts]

        total = len(texts)
        CHUNK_SIZE = 10_000  # Обрабатываем по 10K строк за раз

        result_chunks = []
        for chunk_start in range(0, total, CHUNK_SIZE):
            chunk_end = min(chunk_start + CHUNK_SIZE, total)
            chunk = texts[chunk_start:chunk_end]

            # Кодируем чанк и добавляем в результат
            encoded = [self.encode(t) for t in chunk]
            result_chunks.append(np.array(encoded, dtype=np.int32))

            # Прогресс-бар в одну строку (перезаписываем через \r)
            pct = chunk_end / total * 100
            print(f"\r  Токенизация: {chunk_end:,}/{total:,} строк ({pct:.1f}%)", end='', flush=True)

        print()  # Перенос строки после завершения прогресс-бара

        # Объединяем все чанки в один массив
        return np.concatenate(result_chunks, axis=0)  # shape: (total, seq_length)

    def decode(self, ids):
        """Декодирует список int → строку, пропуская PAD-токены (id=0).

        Args:
            ids: list[int] | np.ndarray | tf.Tensor
        Returns:
            str — декодированный текст
        """
        # Поддержка tf.Tensor и numpy array
        if hasattr(ids, 'numpy'):
            ids = ids.numpy()

        # Фильтруем PAD (0) и конвертируем в int (на случай np.int32)
        ids = [int(i) for i in ids if int(i) != self.pad_id]

        return self.sp.decode(ids)

    def vocabulary_size(self):
        """Возвращает реальный размер словаря BPE модели."""
        return self.sp.get_piece_size()


# ---- Создаём экземпляр токенизатора ----
# seq_length = CONTEXT_WIN + 1: +1 нужен для сдвига X/y (input → target)
tokenizer = SPTokenizer(sp, seq_length=CONTEXT_WIN + 1)

print(f"Токенизатор готов:")
print(f"  Vocab size:  {tokenizer.vocabulary_size():,}")
print(f"  Seq length:  {tokenizer.seq_length} (CONTEXT_WIN={CONTEXT_WIN} + 1 для сдвига)")
print(f"  PAD id:      {tokenizer.pad_id}")

# Быстрая проверка encode / decode round-trip
_test = "The quick brown fox jumps over the lazy dog."
_ids  = tokenizer.encode(_test)
_dec  = tokenizer.decode(_ids)
print(f"\nRound-trip тест:")
print(f"  Вход:    '{_test}'")
print(f"  IDs:      {_ids[:10]}... (длина {len(_ids)})")
print(f"  Декод.:  '{_dec}'")


In [ ]:
# ========================== tf.data PIPELINE ==========================
# Токенизируем весь корпус и создаём эффективный пайплайн для GPU.

print("Токенизация корпуса (может занять ~30 сек)...")
all_token_ids = tokenizer.encode_batch(train_text)
print(f"Токенизировано: {all_token_ids.shape}")

# Разделяем на input (X) и target (y) со сдвигом на 1 токен
X_all = all_token_ids[:, :-1]  # Все токены кроме последнего
y_all = all_token_ids[:, 1:]   # Все токены кроме первого

# Train / Validation split
split_idx = int(len(X_all) * (1 - VALIDATION_SPLIT))
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_val, y_val = X_all[split_idx:], y_all[split_idx:]

# tf.data.Dataset с батчированием и prefetch (GPU не ждёт CPU)
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=min(len(X_train), 50000))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Train: {len(X_train):,} samples | Validation: {len(X_val):,} samples")
print(f"Batches per epoch: {len(X_train) // BATCH_SIZE:,}")

In [ ]:
# ========================== МЕТРИКА ==========================

def perplexity(y_true, y_pred):
    """Perplexity = exp(средняя кросс-энтропия).
    Чем ниже, тем лучше. Идеальная модель: 1. Случайная: vocab_size."""
    return tf.exp(tf.reduce_mean(
        losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    ))


# ========================== TOKEN + POSITION EMBEDDING ==========================

class TokenPositionEmbedding(layers.Layer):
    """Сумма обучаемых token и position эмбеддингов (как в GPT-2).
    Масштабирование: embed × sqrt(d_model) стабилизирует начало обучения."""

    def __init__(self, context_win, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embed = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embed = layers.Embedding(input_dim=context_win, output_dim=embed_dim)
        self.embed_dim = embed_dim

    def call(self, x):
        seq_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        token_emb = self.token_embed(x) * tf.math.sqrt(tf.cast(self.embed_dim, tf.float32))
        return token_emb + self.position_embed(positions)

In [ ]:
# ========================== TRANSFORMER BLOCK (Pre-Norm) ==========================

class TransformerBlock(layers.Layer):
    """Один блок трансформера с Pre-Norm архитектурой (GPT-2+).

    Pre-Norm: LayerNorm ПЕРЕД attention/FFN (не после).
    Преимущество: стабильнее градиенты → можно обучать глубокие модели (6+ блоков)
    без warmup. Используется в GPT-2, GPT-3, LLaMA.
    """

    def __init__(self, embed_dim, heads, feed_forward, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=heads, key_dim=embed_dim // heads
        )
        self.ffn = models.Sequential([
            layers.Dense(feed_forward, activation='gelu'),  # GELU — стандарт GPT-2/3
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Pre-Norm Attention
        normed = self.norm1(inputs)
        attn_output = self.attention(normed, normed, use_causal_mask=True)
        attn_output = self.drop1(attn_output, training=training)
        x = inputs + attn_output  # Residual

        # Pre-Norm FFN
        normed2 = self.norm2(x)
        ffn_output = self.ffn(normed2)
        ffn_output = self.drop2(ffn_output, training=training)
        return x + ffn_output  # Residual

In [ ]:
# ========================== МОДЕЛЬ (GPT-style LLM с Weight Tying) ==========================

class LLM(models.Model):
    """Decoder-only Transformer с Weight Tying.

    Архитектура (V0.14):
    Token+Pos Embed → 6× TransformerBlock(Pre-Norm) → LayerNorm → Embedding^T

    Weight Tying: выходная проекция = транспонированная матрица эмбеддингов.
    Экономит vocab_size × embed_dim параметров (8000 × 256 = 2M в нашем случае).
    """

    def __init__(self, context_win, vocab_size, embed_dim, heads,
                 feed_forward, num_blocks, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_layer = TokenPositionEmbedding(context_win, vocab_size, embed_dim)
        self.blocks = [
            TransformerBlock(embed_dim, heads, feed_forward, dropout_rate)
            for _ in range(num_blocks)
        ]
        self.final_norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        x = self.embed_layer(inputs)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.final_norm(x)
        # Weight Tying: logits = x @ Embedding^T
        logits = tf.matmul(x, self.embed_layer.token_embed.embeddings, transpose_b=True)
        return logits

In [ ]:
# ========================== ФУНКЦИЯ ГЕНЕРАЦИИ (V0.15) ==========================

def generate(model, prompt, max_tokens=80, temperature=0.8, top_k=15,
             top_p=0.9, repetition_penalty=1.3, ngram_block=3):
    """Генерирует текст авторегрессивно с top-k + top-p sampling.

    Улучшения V0.15:
    - N-gram blocking: запрет повторения триграмм (или другого размера)
    - Top-p (nucleus) sampling: динамический выбор числа кандидатов
    - Усиленный repetition penalty (1.3)
    - SentencePiece детокенизация (нет [UNK] проблемы)

    Args:
        model: обученная LLM модель
        prompt: начальный текст
        max_tokens: максимум генерируемых токенов
        temperature: 0.1=детерминированно, 1.5=хаотично
        top_k: первичная фильтрация до k кандидатов
        top_p: nucleus sampling — суммарная вероятность порога (0.9 = 90%)
        repetition_penalty: штраф за повторы (>1.0)
        ngram_block: размер n-граммы для блокировки (3 = триграмм)

    Returns:
        Сгенерированный текст
    """
    # Кодируем промпт через SentencePiece (без паддинга)
    tokens = sp.encode(prompt, out_type=int)

    for _ in range(max_tokens):
        # Скользящее окно: берём последние CONTEXT_WIN токенов
        context = tokens[-CONTEXT_WIN:]

        # Паддим слева если контекст короче CONTEXT_WIN
        if len(context) < CONTEXT_WIN:
            padded = [0] * (CONTEXT_WIN - len(context)) + context
        else:
            padded = context

        input_tensor = tf.convert_to_tensor([padded])
        logits = model(input_tensor, training=False)
        next_logits = logits[0, -1, :].numpy()

        # ---- Жёсткий запрет PAD и UNK ----
        next_logits[0] = -float('inf')  # [PAD]
        next_logits[1] = -float('inf')  # [UNK]

        # ---- Repetition Penalty на последние 50 токенов ----
        for tok_id in set(tokens[-50:]):
            if next_logits[tok_id] < 0:
                next_logits[tok_id] *= repetition_penalty
            else:
                next_logits[tok_id] /= repetition_penalty

        # ---- N-gram Blocking: запрещаем повторение n-грамм ----
        # Если последние (ngram_block-1) токенов уже встречались как начало
        # n-граммы ранее в тексте, блокируем токен, который следовал за ними.
        if ngram_block > 1 and len(tokens) >= ngram_block:
            ngram_prefix = tuple(tokens[-(ngram_block - 1):])
            for i in range(len(tokens) - ngram_block):
                window = tuple(tokens[i : i + ngram_block - 1])
                if window == ngram_prefix:
                    blocked_id = tokens[i + ngram_block - 1]
                    next_logits[blocked_id] = -float('inf')

        # ---- Temperature scaling ----
        next_logits = next_logits / (temperature + 1e-7)

        # ---- Top-k filtering ----
        top_values, top_indices = tf.math.top_k(
            tf.convert_to_tensor(next_logits), k=top_k
        )
        top_probs = tf.nn.softmax(top_values).numpy()

        # ---- Top-p (Nucleus) filtering ----
        # Сортируем вероятности по убыванию и берём только те,
        # чья накопленная сумма ≤ top_p (например, 90%)
        sorted_idx = np.argsort(-top_probs)
        sorted_probs = top_probs[sorted_idx]
        cumulative = np.cumsum(sorted_probs)

        # Находим порог: минимальное число токенов, суммарная вероятность которых ≥ top_p
        cutoff = np.searchsorted(cumulative, top_p) + 1
        cutoff = max(cutoff, 2)  # Минимум 2 кандидата для разнообразия

        # Обнуляем всё за пределами nucleus
        nucleus_idx = sorted_idx[:cutoff]
        nucleus_probs = sorted_probs[:cutoff]
        nucleus_probs = nucleus_probs / nucleus_probs.sum()  # Ренормализация

        # Выбираем из nucleus
        chosen_local = np.random.choice(nucleus_idx, p=nucleus_probs)
        chosen_id = int(top_indices.numpy()[chosen_local])

        # Остановка при endseq или [END]
        piece = sp.id_to_piece(chosen_id)
        if piece == 'endseq' or chosen_id == sp.eos_id():
            break

        tokens.append(chosen_id)

    # Декодируем всю последовательность
    return sp.decode(tokens)

In [ ]:
# ========================== ОБУЧЕНИЕ (V0.15) ==========================

# ---- LR Schedule с Warmup + Cosine Decay ----
# Первые WARMUP_STEPS шагов: LR линейно растёт 0 → LEARNING_RATE
# Затем: плавно падает LEARNING_RATE → MIN_LR по косинусу
# Это СТАНДАРТ для трансформеров (GPT-2, GPT-3, LLaMA все используют warmup).

WARMUP_STEPS = 1000  # ~2-3 эпохи на нашем датасете

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Warmup + Cosine Decay schedule (как в GPT-2/3)."""
    def __init__(self, learning_rate, warmup_steps, total_steps, min_lr):
        super().__init__()
        self.lr = learning_rate
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        # Фаза 1: линейный разогрев
        warmup_lr = self.lr * (step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0))
        # Фаза 2: косинусное затухание
        decay_step = tf.maximum(step - self.warmup_steps, 0.0)
        decay_total = tf.maximum(tf.cast(self.total_steps - self.warmup_steps, tf.float32), 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.lr - self.min_lr) * (
            1 + tf.cos(np.pi * decay_step / decay_total)
        )
        # Выбираем: warmup если step < warmup_steps, иначе cosine
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'learning_rate': self.lr, 'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps, 'min_lr': self.min_lr
        }

# Создаём модель
model = LLM(
    context_win=CONTEXT_WIN,
    vocab_size=actual_vocab_size,
    embed_dim=EMBED_DIM,
    heads=HEADS,
    feed_forward=FEED_FORWARD,
    num_blocks=TRANSFORMER_BLOCKS,
    dropout_rate=DROPOUT_RATE
)

# Подсчёт шагов
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

lr_schedule = WarmupCosineDecay(
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_steps,
    min_lr=MIN_LR
)

# Adam с GRADIENT CLIPPING (clipnorm=1.0)
# Без клиппинга градиенты могут "взорваться" в глубоких моделях (6 блоков),
# что вызывает NaN loss или нестабильное обучение.
optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9,
    clipnorm=1.0      # НОВОЕ: ограничиваем норму градиента
)

model.compile(
    optimizer=optimizer,
    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[perplexity]
)

# Callbacks
model_callbacks = [
    # Сохраняем лучшую модель по val_loss
    callbacks.ModelCheckpoint(
        filepath='best_model_v015.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    # Ранняя остановка если val_loss не улучшается 5 эпох
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# Запуск обучения
print(f"Начинаем обучение: {EPOCHS} эпох, {steps_per_epoch} шагов/эпоху")
print(f"LR: 0 → {LEARNING_RATE} (warmup {WARMUP_STEPS} шагов) → {MIN_LR} (cosine decay)")
print(f"Gradient clipping: clipnorm=1.0")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=model_callbacks,
    verbose=1
)

print(f"\nОбучение завершено!")
print(f"Лучший val_loss: {min(history.history['val_loss']):.4f}")
print(f"Лучший val_perplexity: {min(history.history['val_perplexity']):.2f}")

# Summary модели
model.summary()

In [ ]:
# ========================== БЕСКОНЕЧНЫЙ ЦИКЛ ГЕНЕРАЦИИ (V0.15) ==========================
# Команды:
#   exit/quit/q  — выход
#   settings     — изменить все параметры генерации (включая top_p, ngram_block)
#   Любой текст  — генерация продолжения

print("=" * 60)
print("  SimpleLLM V0.15 — Interactive Text Generation")
print("  BPE | Pre-Norm | Weight Tying | N-gram Block | Top-p")
print("  Команды: 'exit'/'quit'/'q' — выход")
print("           'settings' — изменить параметры генерации")
print("=" * 60)

gen_settings = {
    'max_tokens': 80,
    'temperature': 0.8,
    'top_k': 15,
    'top_p': 0.9,
    'repetition_penalty': 1.3,
    'ngram_block': 3
}

while True:
    prompt = input("\n[Prompt] > ").strip()

    if prompt.lower() in ('exit', 'quit', 'q', ''):
        print("Генерация завершена.")
        break

    if prompt.lower() == 'settings':
        print(f"\nТекущие настройки: {gen_settings}")
        try:
            gen_settings['max_tokens'] = int(input(f"  max_tokens [{gen_settings['max_tokens']}]: ") or gen_settings['max_tokens'])
            gen_settings['temperature'] = float(input(f"  temperature [{gen_settings['temperature']}]: ") or gen_settings['temperature'])
            gen_settings['top_k'] = int(input(f"  top_k [{gen_settings['top_k']}]: ") or gen_settings['top_k'])
            gen_settings['top_p'] = float(input(f"  top_p [{gen_settings['top_p']}]: ") or gen_settings['top_p'])
            gen_settings['repetition_penalty'] = float(input(f"  rep_penalty [{gen_settings['repetition_penalty']}]: ") or gen_settings['repetition_penalty'])
            gen_settings['ngram_block'] = int(input(f"  ngram_block [{gen_settings['ngram_block']}]: ") or gen_settings['ngram_block'])
            print(f"  Обновлено: {gen_settings}")
        except ValueError:
            print("  Ошибка ввода, настройки не изменены.")
        continue

    result = generate(
        model, prompt,
        max_tokens=gen_settings['max_tokens'],
        temperature=gen_settings['temperature'],
        top_k=gen_settings['top_k'],
        top_p=gen_settings['top_p'],
        repetition_penalty=gen_settings['repetition_penalty'],
        ngram_block=gen_settings['ngram_block']
    )

    print(f"\n[Generated] {result}")